# 🔬 CRC-Scan Model Training (STARC-9)
**Target: >97.81% accuracy**

1. Upload `STANFORD-CRC-HE-VAL-SMALL-NORMALIZED.zip` to Google Drive root
2. Set Runtime → GPU (T4)
3. Run all cells (~2-3 hours)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Extract dataset
import zipfile, os
zip_path = '/content/drive/MyDrive/STANFORD-CRC-HE-VAL-SMALL-NORMALIZED.zip'
if not os.path.exists('/content/dataset'):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/dataset')
# Find the actual data folder
for root, dirs, files in os.walk('/content/dataset'):
    if 'ADI' in dirs and 'TUM' in dirs:
        DATA_PATH = root
        break
print(f'Dataset at: {DATA_PATH}')
for d in sorted(os.listdir(DATA_PATH)):
    p = os.path.join(DATA_PATH, d)
    if os.path.isdir(p):
        print(f'  {d}: {len(os.listdir(p))} images')

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ══════════════════════════════════════════
# CNN Model (CRC-SE-CBAM-ASPP)
# ══════════════════════════════════════════
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean', label_smoothing=0.1):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.label_smoothing = label_smoothing
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.alpha, label_smoothing=self.label_smoothing, reduction='none')
        pt = torch.exp(-ce)
        fl = (1 - pt) ** self.gamma * ce
        return fl.mean() if self.reduction == 'mean' else fl.sum() if self.reduction == 'sum' else fl

class ChannelAttention(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        mid = max(ch // r, 4)
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.mx = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(nn.Linear(ch, mid, bias=False), nn.ReLU(True), nn.Linear(mid, ch, bias=False))
        self.sig = nn.Sigmoid()
    def forward(self, x):
        b, c = x.size(0), x.size(1)
        return x * self.sig(self.fc(self.avg(x).view(b,c)) + self.fc(self.mx(x).view(b,c))).view(b,c,1,1)

class SpatialAttention(nn.Module):
    def __init__(self, ks=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, ks, padding=ks//2, bias=False)
        self.sig = nn.Sigmoid()
    def forward(self, x):
        return x * self.sig(self.conv(torch.cat([x.mean(1,keepdim=True), x.max(1,keepdim=True).values], 1)))

class CBAM(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        self.ca = ChannelAttention(ch, r)
        self.sa = SpatialAttention()
    def forward(self, x): return self.sa(self.ca(x))

class ResidualBlock(nn.Module):
    def __init__(self, inc, outc, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(inc, outc, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(outc)
        self.conv2 = nn.Conv2d(outc, outc, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(outc)
        self.cbam = CBAM(outc)
        self.shortcut = nn.Sequential() if (stride==1 and inc==outc) else nn.Sequential(nn.Conv2d(inc,outc,1,stride=stride,bias=False), nn.BatchNorm2d(outc))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), True)
        out = self.cbam(self.bn2(self.conv2(out)))
        return F.relu(out + self.shortcut(x), True)

class ASPPModule(nn.Module):
    def __init__(self, inc, outc=256):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(inc,outc,1,bias=False), nn.BatchNorm2d(outc), nn.ReLU(True)),
            nn.Sequential(nn.Conv2d(inc,outc,1,bias=False), nn.BatchNorm2d(outc), nn.ReLU(True)),
            nn.Sequential(nn.Conv2d(inc,outc,3,padding=6,dilation=6,bias=False), nn.BatchNorm2d(outc), nn.ReLU(True)),
            nn.Sequential(nn.Conv2d(inc,outc,3,padding=12,dilation=12,bias=False), nn.BatchNorm2d(outc), nn.ReLU(True)),
        ])
        self.project = nn.Sequential(nn.Conv2d(outc*4,outc,1,bias=False), nn.BatchNorm2d(outc), nn.ReLU(True), nn.Dropout2d(0.3))
    def forward(self, x):
        h, w = x.shape[2:]
        outs = []
        for i, b in enumerate(self.branches):
            f = b(x)
            if i == 0: f = F.interpolate(f, (h,w), mode='bilinear', align_corners=False)
            outs.append(f)
        return self.project(torch.cat(outs, 1))

class Model(nn.Module):
    def __init__(self, num_classes=9):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3,32,7,stride=2,padding=3,bias=False), nn.BatchNorm2d(32), nn.ReLU(True))
        self.stage1 = nn.Sequential(ResidualBlock(32,64,2), ResidualBlock(64,64))
        self.stage2 = nn.Sequential(ResidualBlock(64,128,2), ResidualBlock(128,128))
        self.stage3 = nn.Sequential(ResidualBlock(128,256,2), ResidualBlock(256,256))
        self.stage4 = nn.Sequential(ResidualBlock(256,512,2), ResidualBlock(512,512))
        self.stage5 = nn.Sequential(ResidualBlock(512,1024,2), ResidualBlock(1024,1024))
        self.aspp = ASPPModule(1024, 256)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.flatten = nn.Flatten()
        self.classifier = nn.Sequential(
            nn.Linear(256,512), nn.BatchNorm1d(512), nn.ReLU(True), nn.Dropout(0.5),
            nn.Linear(512,256), nn.BatchNorm1d(256), nn.ReLU(True), nn.Dropout(0.3),
            nn.Linear(256, num_classes))
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)): nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear): nn.init.xavier_normal_(m.weight); m.bias is not None and nn.init.constant_(m.bias,0)
    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x); x = self.stage2(x); x = self.stage3(x)
        x = self.stage4(x); x = self.stage5(x)
        x = self.aspp(x); x = self.gap(x); x = self.flatten(x)
        return self.classifier(x)

print('Model defined. Params:', f"{sum(p.numel() for p in Model(9).parameters()):,}")

In [ ]:
# ══════════════════════════════════════════
# Dataset & Training
# ══════════════════════════════════════════
import numpy as np, time, json, os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from tqdm.auto import tqdm

NUM_CLASSES = 9
CLASS_NAMES = ['ADI','LYM','MUC','MUS','NCS','NOR','BLD','FCT','TUM']
LABEL_MAP = {'ADI':0,'LYM':1,'MUC':2,'MUS':3,'NCS':4,'NOR':5,'BLD':6,'FCT':7,'TUM':8,
             'Adipose':0,'Lymphocyte':1,'Mucin':2,'Muscle':3,'Necrotic_debris':4,
             'Normal':5,'Red_blood_cells':6,'Loose_connective_tissue':7,'Tumor':8}
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(256, scale=(0.7,1.0), ratio=(0.85,1.15)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(180),
    transforms.RandomAffine(0, translate=(0.1,0.1), shear=10),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.15, scale=(0.02,0.12)),
])
val_tf = transforms.Compose([transforms.Resize((256,256)), transforms.ToTensor(), transforms.Normalize(MEAN,STD)])

class CRCDataset(Dataset):
    def __init__(self, path, transform):
        self.files, self.labels, self.tf = [], [], transform
        for name, idx in LABEL_MAP.items():
            d = os.path.join(path, name)
            if not os.path.isdir(d): continue
            for f in os.listdir(d):
                if f.lower().endswith(('.png','.jpg','.jpeg')):
                    self.files.append(os.path.join(d,f)); self.labels.append(idx)
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert('RGB')
        return self.tf(img), self.labels[i]

train_ds = CRCDataset(DATA_PATH, train_tf)
val_ds = CRCDataset(DATA_PATH, val_tf)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')

In [ ]:
# ══════════════════════════════════════════
# TRAIN (50 epochs, GPU)
# ══════════════════════════════════════════
device = torch.device('cuda')
model = Model(NUM_CLASSES).to(device)

# Class-balanced Focal Loss
labels_arr = np.array(train_ds.labels)
weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=labels_arr)
class_weights = torch.FloatTensor(weights).to(device)
criterion = FocalLoss(alpha=class_weights, gamma=2.0, label_smoothing=0.1)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3, steps_per_epoch=len(train_loader),
    epochs=50, pct_start=0.3, anneal_strategy='cos', div_factor=10, final_div_factor=1000)

best_f1, best_acc = 0, 0
history = []

for epoch in range(50):
    model.train()
    running_loss, preds_all, labels_all = 0, [], []
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/50')
    for imgs, lbls in pbar:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item() * imgs.size(0)
        preds_all.extend(out.argmax(1).cpu().numpy())
        labels_all.extend(lbls.cpu().numpy())
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    
    train_acc = accuracy_score(labels_all, preds_all)
    train_f1 = f1_score(labels_all, preds_all, average='macro')
    
    # Validation
    model.eval()
    vp, vl = [], []
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs = imgs.to(device)
            logits = model(imgs)
            vp.extend(logits.argmax(1).cpu().numpy())
            vl.extend(lbls.numpy())
    val_acc = accuracy_score(vl, vp)
    val_f1 = f1_score(vl, vp, average='macro')
    
    history.append({'epoch':epoch+1, 'train_acc':train_acc, 'train_f1':train_f1, 'val_acc':val_acc, 'val_f1':val_f1})
    print(f'  Train Acc: {train_acc:.4f} F1: {train_f1:.4f} | Val Acc: {val_acc:.4f} F1: {val_f1:.4f}')
    
    if val_f1 > best_f1:
        best_f1, best_acc = val_f1, val_acc
        torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch+1,
                    'val_acc': val_acc, 'val_f1': val_f1}, '/content/best_custom_cnn.pth')
        print(f'  ✅ SAVED (Val F1: {val_f1:.4f}, Acc: {val_acc:.4f})')

print(f'\n🏆 Best: Acc={best_acc:.4f}, F1={best_f1:.4f}')

In [ ]:
# Save to Google Drive
import shutil
dst = '/content/drive/MyDrive/best_custom_cnn.pth'
shutil.copy('/content/best_custom_cnn.pth', dst)
print(f'Weights saved to Drive: {dst}')
print(f'Copy this file to: weights/Model_weights/best_custom_cnn.pth')

In [ ]:
# Final comparison vs paper
print('=' * 60)
print('  RESULTS vs STARC-9 Paper')
print('=' * 60)
print(f'  Paper CNN:  Acc=97.81%  F1=98.12%')
print(f'  Ours:       Acc={best_acc*100:.2f}%  F1={best_f1*100:.2f}%')
v = '✅ BEATEN!' if best_acc > 0.9781 else '❌ Not yet'
print(f'  Verdict: {v}')
print('=' * 60)